In [2]:
import pandas as pd
import numpy as np
import networkx as nx
from io import StringIO
import urllib.request
import gzip
import os

def prepare_datasets():
    os.makedirs("data", exist_ok=True)
    
    # ------------------------------------------------------------
    # 1. SACHS DATASET
    # ------------------------------------------------------------
    SACHS_URL = "https://www.bnlearn.com/book-crc/code/sachs.interventional.txt.gz"
    SACHS_VARIABLES = [
        "Raf", "Mek", "Plcg", "PIP2", "PIP3",
        "Erk", "Akt", "PKA", "PKC", "Jnk", "p38"
    ]
    
    print("[Phase 1] Downloading Sachs dataset...")
    try:
        with urllib.request.urlopen(SACHS_URL) as response:
            compressed = response.read()
        decompressed = gzip.decompress(compressed).decode("utf-8")
        df_sachs = pd.read_csv(StringIO(decompressed), delimiter=" ")
        
        # Normalize column names
        df_sachs.columns = df_sachs.columns.str.strip()
        df_sachs = df_sachs.rename(columns={c: "p38" for c in df_sachs.columns if c.lower() == "p38"})
        
        # Separate Observational and Interventional
        df_obs = df_sachs[df_sachs["INT"] == 0][SACHS_VARIABLES].dropna().reset_index(drop=True)
        df_int = df_sachs[df_sachs["INT"] == 1][SACHS_VARIABLES].dropna().reset_index(drop=True)
        
        # Save to disk
        df_obs.to_csv("data/sachs_obs.csv", index=False)
        df_int.to_csv("data/sachs_int.csv", index=False)
        
        print(f"[Phase 1] Sachs Observational: {len(df_obs)} rows saved to data/sachs_obs.csv")
        print(f"[Phase 1] Sachs Interventional: {len(df_int)} rows saved to data/sachs_int.csv")
        
    except Exception as e:
        print(f"[Phase 1] Sachs download failed: {e}")
        return
        
    # ------------------------------------------------------------
    # 2. ASIA DATASET
    # ------------------------------------------------------------
    print("\n[Phase 1] Generating Asia dataset from CPTs...")
    np.random.seed(42)
    n_samples = 5000

    asia = np.random.binomial(1, 0.01, n_samples)
    smoke = np.random.binomial(1, 0.5, n_samples)
    tub = np.where(asia == 1, np.random.binomial(1, 0.05, n_samples), np.random.binomial(1, 0.01, n_samples))
    lung = np.where(smoke == 1, np.random.binomial(1, 0.10, n_samples), np.random.binomial(1, 0.01, n_samples))
    bronc = np.where(smoke == 1, np.random.binomial(1, 0.60, n_samples), np.random.binomial(1, 0.30, n_samples))
    either = np.logical_or(tub, lung).astype(int)
    xray = np.where(either == 1, np.random.binomial(1, 0.98, n_samples), np.random.binomial(1, 0.05, n_samples))
    
    p_dysp = np.where(
        (either == 1) & (bronc == 1), 0.90,
        np.where((either == 1) & (bronc == 0), 0.70,
                 np.where((either == 0) & (bronc == 1), 0.40, 0.10)))
    dysp = np.random.binomial(1, p_dysp, n_samples)

    df_asia = pd.DataFrame({
        "asia": asia, "smoke": smoke, "tub": tub, "lung": lung,
        "bronc": bronc, "either": either, "xray": xray, "dysp": dysp
    })
    
    df_asia.to_csv("data/asia.csv", index=False)
    print(f"[Phase 1] Asia generated: {len(df_asia)} rows saved to data/asia.csv")

    print("\n[Phase 1 Complete] All datasets are ready in the 'data/' folder.")

# Run the preparation
prepare_datasets()

[Phase 1] Downloading Sachs dataset...
[Phase 1] Sachs Observational: 1800 rows saved to data/sachs_obs.csv
[Phase 1] Sachs Interventional: 0 rows saved to data/sachs_int.csv

[Phase 1] Generating Asia dataset from CPTs...
[Phase 1] Asia generated: 5000 rows saved to data/asia.csv

[Phase 1 Complete] All datasets are ready in the 'data/' folder.


In [5]:
import pandas as pd
import numpy as np

def create_heterogeneous_clients(dataset_path, K, alpha, missing_rate, seed=42):
    """
    Splits dataset horizontally (rows) into K clients.
    Splits dataset vertically (variables) so each client only sees a fraction `alpha`.
    Guarantees that all global variables are covered by at least one client.
    """
    df = pd.read_csv(dataset_path)
    all_vars = df.columns.tolist()
    num_vars = len(all_vars)
    
    # Calculate how many variables each client should see
    vars_per_client = max(1, int(num_vars * alpha))
    
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(df))
    splits = np.array_split(idx, K)
    
    # Shuffle variables to ensure random distribution across clients
    rng_vars = np.random.RandomState(seed + 1)
    shuffled_vars = rng_vars.permutation(all_vars).tolist()
    
    clients = []
    
    for k, split in enumerate(splits):
        # 1. Horizontal Split (Rows)
        cdf = df.iloc[split].reset_index(drop=True)
        
        # 2. Vertical Split (Variables)
        # Use round-robin to guarantee all variables are assigned across clients
        selected_vars = []
        for i in range(vars_per_client):
            var_idx = (k * vars_per_client + i) % num_vars
            selected_vars.append(shuffled_vars[var_idx])
            
        cdf = cdf[selected_vars]
        
        # 3. Inject Missingness (MCAR)
        if missing_rate > 0:
            client_rng = np.random.RandomState(seed + k)
            out = cdf.copy().astype(float)
            for col in out.columns:
                mask = client_rng.rand(len(out)) < missing_rate
                out.loc[mask, col] = np.nan
            cdf = out
            
        clients.append(cdf)
        
    return clients, all_vars

# ------------------------------------------------------------
# Run for alpha = 2/11 (10% missingness)
# ------------------------------------------------------------
print("=== Simulating Baseline Variable Heterogeneity ===")
print("Alpha = 2/11 (Each client only holds 2 variables out of 11)")

K = 6
alpha = 2 / 11
missing_rate = 0.10

clients, all_vars = create_heterogeneous_clients("data/sachs_obs.csv", K, alpha, missing_rate)

for i, cdf in enumerate(clients):
    print(f"Client {i+1} Variables: {cdf.columns.tolist()} | Shape: {cdf.shape}")

# ------------------------------------------------------------
# Global Coverage Check
# ------------------------------------------------------------
print("\n=== Checking Global Variable Coverage ===")
observed_union = set()
for cdf in clients:
    observed_union.update(cdf.columns.tolist())

missing_globally = set(all_vars) - observed_union

print(f"Total variables in dataset: {len(all_vars)}")
print(f"Variables seen by at least one client: {len(observed_union)}")
if missing_globally:
    print(f"Variables completely missing globally: {missing_globally}")
else:
    print("Success: All 11 variables are observed by at least one client.")

=== Simulating Baseline Variable Heterogeneity ===
Alpha = 2/11 (Each client only holds 2 variables out of 11)
Client 1 Variables: ['p38', 'PKA'] | Shape: (300, 2)
Client 2 Variables: ['Akt', 'PKC'] | Shape: (300, 2)
Client 3 Variables: ['PIP2', 'Jnk'] | Shape: (300, 2)
Client 4 Variables: ['Plcg', 'Erk'] | Shape: (300, 2)
Client 5 Variables: ['Mek', 'Raf'] | Shape: (300, 2)
Client 6 Variables: ['PIP3', 'p38'] | Shape: (300, 2)

=== Checking Global Variable Coverage ===
Total variables in dataset: 11
Variables seen by at least one client: 11
Success: All 11 variables are observed by at least one client.


In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

def compute_client_weights(client_df, client_id):
    """
    Computes the IPW weights and effective sample size for a single client.
    """
    V_m = [col for col in client_df.columns if client_df[col].isna().mean() >= 0.05]
    V_o = [col for col in client_df.columns if client_df[col].isna().mean() < 0.05]
    
    # Initialize weights to 1.0 for all rows
    w = np.ones(len(client_df))
    
    if V_m:
        for target in V_m:
            # Use V_o as parents. If V_o is empty, use all other V_m variables.
            parents = V_o if V_o else [c for c in client_df.columns if c != target]
            
            # Create missingness indicator
            R = client_df[target].isna().astype(int).values
            
            # Restrict to rows where parents are fully observed
            pdata = client_df[parents]
            mask = pdata.notna().all(axis=1).values
            
            if mask.sum() > 10 and R[mask].sum() > 5:
                X = pdata.values[mask].astype(float)
                R_obs = R[mask]
                
                # Fit logistic regression
                scaler = StandardScaler()
                Xs = scaler.fit_transform(X)
                model = LogisticRegression(penalty="l2", C=1.0, max_iter=1000, random_state=42)
                model.fit(Xs, R_obs)
                
                # Predict probability of observation P(R=0 | parents)
                probs_missing = model.predict_proba(Xs)[:, 1]
                probs_obs = 1 - probs_missing
                
                # Compute weight: w = 1 / P(R=0 | parents)
                # Clip weights at 95th percentile to prevent extreme values
                weights_col = 1.0 / np.clip(probs_obs, 1e-6, 1.0)
                clip_val = np.percentile(weights_col, 95)
                weights_col = np.minimum(weights_col, clip_val)
                
                # Update the row weights only for the rows we used
                w[mask] *= weights_col
                
    # Normalize weights so their mean is 1
    w = w / w.mean()
    
    # Compute effective sample size
    n_eff = (w.sum() ** 2) / (w ** 2).sum()
    
    return w, n_eff

# ------------------------------------------------------------
# Execute IPW on the 6 Clients
# ------------------------------------------------------------
print("=== Computing Client-Side IPW Weights ===")

# We use the `clients` list generated from Chunk 2
client_weights = []
client_neffs = []

for i, cdf in enumerate(clients):
    w, n_eff = compute_client_weights(cdf, i)
    client_weights.append(w)
    client_neffs.append(n_eff)
    
    print(f"Client {i+1} ({cdf.columns.tolist()}):")
    print(f"  - Rows: {len(cdf)}")
    print(f"  - Effective Sample Size (n_eff): {n_eff:.1f} ({n_eff/len(cdf):.1%} retained)")
    print(f"  - Max weight: {w.max():.2f}")

=== Computing Client-Side IPW Weights ===
Client 1 (['p38', 'PKA']):
  - Rows: 300
  - Effective Sample Size (n_eff): 299.2 (99.7% retained)
  - Max weight: 1.03
Client 2 (['Akt', 'PKC']):
  - Rows: 300
  - Effective Sample Size (n_eff): 299.2 (99.7% retained)
  - Max weight: 1.05
Client 3 (['PIP2', 'Jnk']):
  - Rows: 300
  - Effective Sample Size (n_eff): 299.4 (99.8% retained)
  - Max weight: 1.03
Client 4 (['Plcg', 'Erk']):
  - Rows: 300
  - Effective Sample Size (n_eff): 299.5 (99.8% retained)
  - Max weight: 1.07
Client 5 (['Mek', 'Raf']):
  - Rows: 300
  - Effective Sample Size (n_eff): 298.5 (99.5% retained)
  - Max weight: 1.08
Client 6 (['PIP3', 'p38']):
  - Rows: 300
  - Effective Sample Size (n_eff): 299.6 (99.9% retained)
  - Max weight: 1.05


In [7]:
def _c3(x, w):
    mu = np.sum(w * x) / np.sum(w)
    return np.sum(w * (x - mu) ** 3) / np.sum(w)

def _c12(x, y, w):
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) * (y - my) ** 2) / np.sum(w)

def _c21(x, y, w):
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) ** 2 * (y - my)) / np.sum(w)

def compute_pair_cumulants(df, w, x, y, min_obs=50):
    """Computes weighted cumulants for a specific pair (x, y)."""
    mask = df[x].notna() & df[y].notna()
    n = mask.sum()
    if n < min_obs: 
        return None
        
    xv = df.loc[mask, x].values.astype(float)
    yv = df.loc[mask, y].values.astype(float)
    
    # Renormalize weights for the observed rows
    ww = w[mask] * (n / w[mask].sum())
    
    return {
        f"C3_{x}": float(_c3(xv, ww)), 
        f"C3_{y}": float(_c3(yv, ww)),
        f"C12_{x}_{y}": float(_c12(xv, yv, ww)), 
        f"C21_{x}_{y}": float(_c21(xv, yv, ww)),
        f"C12_{y}_{x}": float(_c12(yv, xv, ww)), 
        f"C21_{y}_{x}": float(_c21(yv, xv, ww)),
        "n_obs": int(n)
    }

# ------------------------------------------------------------
# Execute Cumulant Computation on the 6 Clients
# ------------------------------------------------------------
print("=== Computing Client-Side Weighted Cumulants ===")

server_received_cumulants = []

for i, cdf in enumerate(clients):
    w = client_weights[i]
    vars_in_client = cdf.columns.tolist()
    
    # Since each client has exactly 2 variables, we compute the pair
    if len(vars_in_client) == 2:
        x, y = vars_in_client[0], vars_in_client[1]
        pair_cums = compute_pair_cumulants(cdf, w, x, y)
        
        if pair_cums:
            server_received_cumulants.append({
                "client_id": i + 1,
                "variables": [x, y],
                "cumulants": pair_cums,
                "n_eff": client_neffs[i]
            })
            print(f"Client {i+1} ({x}, {y}): Sent cumulants to server.")
        else:
            print(f"Client {i+1} ({x}, {y}): Not enough overlapping data.")
            
    else:
        # (If alpha > 2/11, clients might have more than 2 variables)
        # For this specific alpha=2/11 run, all clients have exactly 2 variables.
        pass

=== Computing Client-Side Weighted Cumulants ===
Client 1 (p38, PKA): Sent cumulants to server.
Client 2 (Akt, PKC): Sent cumulants to server.
Client 3 (PIP2, Jnk): Sent cumulants to server.
Client 4 (Plcg, Erk): Sent cumulants to server.
Client 5 (Mek, Raf): Sent cumulants to server.
Client 6 (PIP3, p38): Sent cumulants to server.


In [8]:
import networkx as nx

# ------------------------------------------------------------
# 1. Define Global Skeleton & Ground Truth
# ------------------------------------------------------------
SACHS_TRUE_EDGES = [
    ("Raf", "Mek"), ("Mek", "Plcg"), ("Mek", "PIP2"), ("Mek", "PIP3"),
    ("Plcg", "PIP2"), ("Plcg", "PIP3"), ("PIP3", "PIP2"), ("PIP3", "Akt"),
    ("PIP2", "PKC"), ("PIP3", "PKC"), ("Akt", "PKA"), ("PKC", "PKA"),
    ("PKC", "Jnk"), ("PKA", "Jnk"), ("PKA", "p38"), ("PKA", "Erk"),
    ("Jnk", "p38"), ("Jnk", "Erk"), ("p38", "Erk"),
]

all_vars = ["Raf", "Mek", "Plcg", "PIP2", "PIP3", "Erk", "Akt", "PKA", "PKC", "Jnk", "p38"]

def _skeleton():
    G = nx.Graph()
    G.add_nodes_from(all_vars)
    for u, v in SACHS_TRUE_EDGES: 
        G.add_edge(u, v)
    return G

def _star_graphs(skel):
    out = []
    for c in all_vars:
        if c not in skel: continue
        periph = list(skel.neighbors(c))
        if periph: 
            out.append({"center": c, "peripherals": periph})
    return out

# ------------------------------------------------------------
# 2. Server-Side Aggregation
# ------------------------------------------------------------
print("=== Server Aggregating Cumulants ===")

total_neff = sum([c["n_eff"] for c in server_received_cumulants])
agg_cums = {}

for client_data in server_received_cumulants:
    wt = client_data["n_eff"] / total_neff
    for key, val in client_data["cumulants"].items():
        if key == "n_obs":
            continue
        agg_cums.setdefault(key, 0.0)
        agg_cums[key] += wt * val

print(f"Total effective sample size across clients: {total_neff:.1f}")
print(f"Total unique cumulants aggregated: {len(agg_cums)}")

# ------------------------------------------------------------
# 3. Recursive Source Recovery
# ------------------------------------------------------------
def _tau(cums, x, y):
    c3x = cums.get(f"C3_{x}", 0.0); c3y = cums.get(f"C3_{y}", 0.0)
    c12xy = cums.get(f"C12_{x}_{y}", 0.0); c21xy = cums.get(f"C21_{x}_{y}", 0.0)
    c12yx = cums.get(f"C12_{y}_{x}", 0.0); c21yx = cums.get(f"C21_{y}_{x}", 0.0)
    return (abs(abs(c3x) * c12xy - c21xy * c12yx), abs(abs(c3y) * c12yx - c21yx * c12xy))

def _find_source(cums, vars_, ratio=0.05):
    n = len(vars_)
    tau = {}
    for i in range(n):
        for j in range(n):
            if i == j: continue
            t, _ = _tau(cums, vars_[i], vars_[j])
            tau[(vars_[i], vars_[j])] = t
    if not tau: return None
    thr = ratio * max(tau.values())
    for s in vars_:
        if all(tau.get((s, j), 1.0) <= thr for j in vars_ if j != s):
            return s
    return None

def _eliminate(cums, src, desc):
    upd = dict(cums)
    c3s = cums.get(f"C3_{src}", 0.0)
    if abs(c3s) < 1e-10: return upd
    a = {i: np.clip(cums.get(f"C21_{i}_{src}", 0.0) / c3s, -2, 2) for i in desc if i != src}
    for i in desc:
        if i == src: continue
        upd[f"C3_{i}"] = cums.get(f"C3_{i}", 0.0) - (a.get(i, 0) ** 3) * c3s
    for i in desc:
        if i == src: continue
        for j in desc:
            if j == src or i == j: continue
            ai, aj = a.get(i, 0), a.get(j, 0)
            upd[f"C12_{i}_{j}"] = cums.get(f"C12_{i}_{j}", 0.0) - ai * (aj ** 2) * c3s
            upd[f"C21_{i}_{j}"] = cums.get(f"C21_{i}_{j}", 0.0) - (ai ** 2) * aj * c3s
    return upd

def _recover(cums, vars_, ratio=0.05):
    order, rem, cur = [], list(vars_), dict(cums)
    while len(rem) > 1:
        s = _find_source(cur, rem, ratio)
        if s is None: break
        order.append(s); rem.remove(s)
        if rem: cur = _eliminate(cur, s, rem)
    if len(rem) == 1: order.append(rem[0])
    return order

print("\n=== Recovering Causal Order from Star Graphs ===")
skel = _skeleton()
star_graphs = _star_graphs(skel)

orders = []
for sg in star_graphs:
    c, periph = sg["center"], sg["peripherals"]
    o = _recover(agg_cums, [c] + periph, ratio=0.05)
    if o: 
        orders.append(o)

# ------------------------------------------------------------
# 4. Build Graph and Evaluate
# ------------------------------------------------------------
def _build_graph(orders):
    G = nx.DiGraph()
    G.add_nodes_from(all_vars)
    for o in orders:
        for i in range(len(o) - 1):
            G.add_edge(o[i], o[i + 1])
    return G

def _metrics(G):
    true = nx.DiGraph()
    true.add_nodes_from(all_vars)
    true.add_edges_from(SACHS_TRUE_EDGES)
    est = set(G.edges()); tru = set(true.edges())
    tp = len(est & tru); fp = len(est - tru); fn = len(tru - est)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"shd": len(tru - est) + len(est - tru), "f1": f1}

G = _build_graph(orders)
m = _metrics(G)

print("\n=== FINAL RESULT FOR K=6 (alpha = 2/11) ===")
print(f"F1-orie: {m['f1']:.3f}")
print(f"SHD: {m['shd']}")

=== Server Aggregating Cumulants ===
Total effective sample size across clients: 1795.4
Total unique cumulants aggregated: 35

=== Recovering Causal Order from Star Graphs ===

=== FINAL RESULT FOR K=6 (alpha = 2/11) ===
F1-orie: 0.550
SHD: 18


In [14]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 1. Helper Functions (Client side)
# ------------------------------------------------------------
def create_heterogeneous_clients(dataset_path, K, alpha, missing_rate, seed=42):
    df = pd.read_csv(dataset_path)
    all_vars = df.columns.tolist()
    num_vars = len(all_vars)
    vars_per_client = max(1, int(num_vars * alpha))
    
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(df))
    splits = np.array_split(idx, K)
    
    clients = []
    for k, split in enumerate(splits):
        cdf = df.iloc[split].reset_index(drop=True)
        
        # Deterministic variable split based on k
        start_idx = (k * vars_per_client) % num_vars
        selected_vars = [all_vars[(start_idx + i) % num_vars] for i in range(vars_per_client)]
        cdf = cdf[selected_vars]
        
        if missing_rate > 0:
            client_rng = np.random.RandomState(seed + k)
            out = cdf.copy().astype(float)
            for col in out.columns:
                mask = client_rng.rand(len(out)) < missing_rate
                out.loc[mask, col] = np.nan
            cdf = out
        clients.append(cdf)
    return clients

def compute_client_weights(client_df):
    V_m = [col for col in client_df.columns if client_df[col].isna().mean() >= 0.05]
    V_o = [col for col in client_df.columns if client_df[col].isna().mean() < 0.05]
    w = np.ones(len(client_df))
    
    if V_m:
        for target in V_m:
            parents = V_o if V_o else [c for c in client_df.columns if c != target]
            R = client_df[target].isna().astype(int).values
            pdata = client_df[parents]
            mask = pdata.notna().all(axis=1).values
            
            if mask.sum() > 10 and R[mask].sum() > 5:
                X = pdata.values[mask].astype(float)
                R_obs = R[mask]
                scaler = StandardScaler()
                Xs = scaler.fit_transform(X)
                model = LogisticRegression(penalty="l2", C=1.0, max_iter=1000, random_state=42)
                model.fit(Xs, R_obs)
                
                probs = model.predict_proba(Xs)[:, 1]
                weights_col = 1.0 / np.clip(1 - probs, 1e-6, 1.0)
                clip_val = np.percentile(weights_col, 95)
                weights_col = np.minimum(weights_col, clip_val)
                w[mask] *= weights_col
                    
    w = w / w.mean()
    n_eff = (w.sum() ** 2) / (w ** 2).sum()
    return w, n_eff

def _c3(x, w):
    mu = np.sum(w * x) / np.sum(w)
    return np.sum(w * (x - mu) ** 3) / np.sum(w)

def _c12(x, y, w):
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) * (y - my) ** 2) / np.sum(w)

def _c21(x, y, w):
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) ** 2 * (y - my)) / np.sum(w)

def compute_pair_cumulants(df, w, x, y, min_obs=30):
    mask = df[x].notna() & df[y].notna()
    n = mask.sum()
    if n < min_obs: return None
    xv = df.loc[mask, x].values.astype(float)
    yv = df.loc[mask, y].values.astype(float)
    ww = w[mask] * (n / w[mask].sum())
    
    # RANK TRANSFORMATION (Prevents collapse at high missingness)
    xv = rankdata(xv)
    yv = rankdata(yv)
    
    std_x = np.std(xv) if np.std(xv) > 1e-8 else 1.0
    std_y = np.std(yv) if np.std(yv) > 1e-8 else 1.0
    xv = (xv - np.mean(xv)) / std_x
    yv = (yv - np.mean(yv)) / std_y
    
    return {
        f"C3_{x}": float(_c3(xv, ww)), 
        f"C3_{y}": float(_c3(yv, ww)),
        f"C12_{x}_{y}": float(_c12(xv, yv, ww)), 
        f"C21_{x}_{y}": float(_c21(xv, yv, ww)),
        f"C12_{y}_{x}": float(_c12(yv, xv, ww)), 
        f"C21_{y}_{x}": float(_c21(yv, xv, ww)),
        "n_obs": int(n)
    }

# ------------------------------------------------------------
# 2. Corrected Ground Truth & Server Recovery
# ------------------------------------------------------------
SACHS_TRUE_EDGES = [
    ("Raf", "Mek"), ("Mek", "Plcg"), ("Mek", "PIP2"), ("Mek", "PIP3"),
    ("Plcg", "PIP2"), ("Plcg", "PIP3"), ("PIP3", "PIP2"), ("PIP3", "Akt"),
    ("PIP2", "PKC"), ("PIP3", "PKC"), ("Akt", "PKA"), ("PKC", "PKA"),
    ("PKC", "Jnk"), ("PKA", "Jnk"), ("PKA", "p38"), ("PKA", "Erk"),
    ("Jnk", "p38"), ("Jnk", "Erk"), ("p38", "Erk"),
]
all_vars = ["Raf", "Mek", "Plcg", "PIP2", "PIP3", "Erk", "Akt", "PKA", "PKC", "Jnk", "p38"]

def _skeleton():
    G = nx.Graph()
    G.add_nodes_from(all_vars)
    for u, v in SACHS_TRUE_EDGES: G.add_edge(u, v)
    return G

def _star_graphs(skel):
    out = []
    for c in all_vars:
        if c not in skel: continue
        periph = list(skel.neighbors(c))
        if periph: out.append({"center": c, "peripherals": periph})
    return out

def _tau(cums, x, y):
    c3x = cums.get(f"C3_{x}", 0.0); c3y = cums.get(f"C3_{y}", 0.0)
    c12xy = cums.get(f"C12_{x}_{y}", 0.0); c21xy = cums.get(f"C21_{x}_{y}", 0.0)
    c12yx = cums.get(f"C12_{y}_{x}", 0.0); c21yx = cums.get(f"C21_{y}_{x}", 0.0)
    return (abs(abs(c3x) * c12xy - c21xy * c12yx), abs(abs(c3y) * c12yx - c21yx * c12xy))

def _find_source(cums, vars_, ratio=0.05):
    n = len(vars_)
    tau = {}
    for i in range(n):
        for j in range(n):
            if i == j: continue
            t, _ = _tau(cums, vars_[i], vars_[j])
            tau[(vars_[i], vars_[j])] = t
    if not tau: return None
    thr = ratio * max(tau.values())
    for s in vars_:
        if all(tau.get((s, j), 1.0) <= thr for j in vars_ if j != s):
            return s
    return None

def _eliminate(cums, src, desc):
    upd = dict(cums)
    c3s = cums.get(f"C3_{src}", 0.0)
    if abs(c3s) < 1e-10: return upd
    a = {i: np.clip(cums.get(f"C21_{i}_{src}", 0.0) / c3s, -2, 2) for i in desc if i != src}
    for i in desc:
        if i == src: continue
        upd[f"C3_{i}"] = cums.get(f"C3_{i}", 0.0) - (a.get(i, 0) ** 3) * c3s
    for i in desc:
        if i == src: continue
        for j in desc:
            if j == src or i == j: continue
            ai, aj = a.get(i, 0), a.get(j, 0)
            upd[f"C12_{i}_{j}"] = cums.get(f"C12_{i}_{j}", 0.0) - ai * (aj ** 2) * c3s
            upd[f"C21_{i}_{j}"] = cums.get(f"C21_{i}_{j}", 0.0) - (ai ** 2) * aj * c3s
    return upd

def _recover(cums, vars_, ratio=0.05):
    order, rem, cur = [], list(vars_), dict(cums)
    while len(rem) > 1:
        s = _find_source(cur, rem, ratio)
        if s is None: break
        order.append(s); rem.remove(s)
        if rem: cur = _eliminate(cur, s, rem)
    if len(rem) == 1: order.append(rem[0])
    return order

def _build_graph(orders):
    G = nx.DiGraph()
    G.add_nodes_from(all_vars)
    for o in orders:
        for i in range(len(o) - 1):
            G.add_edge(o[i], o[i + 1])
    return G

def _metrics(G):
    true = nx.DiGraph()
    true.add_nodes_from(all_vars)
    true.add_edges_from(SACHS_TRUE_EDGES)
    est = set(G.edges()); tru = set(true.edges())
    tp = len(est & tru); fp = len(est - tru); fn = len(tru - est)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"shd": len(tru - est) + len(est - tru), "f1": f1}

# ------------------------------------------------------------
# 3. Full Loop
# ------------------------------------------------------------
alphas = [2/11, 3/11, 4/11, 5/11, 6/11]
missing_rates = [0.10, 0.20, 0.30, 0.40, 0.50]

print("=== RUNNING ALL ALPHA DISTRIBUTIONS FOR K=6 ===")
results = []
skel = _skeleton()
star_graphs = _star_graphs(skel)

for alpha, m_rate in zip(alphas, missing_rates):
    print(f"\nRunning alpha = {alpha:.2f}, Missingness: {m_rate}")
    
    clients = create_heterogeneous_clients("data/sachs_obs.csv", K=6, alpha=alpha, missing_rate=m_rate)
    
    server_cumulants = []
    for i, cdf in enumerate(clients):
        w, n_eff = compute_client_weights(cdf)
        vars_in_client = cdf.columns.tolist()
        
        client_cums = {}
        for x_idx in range(len(vars_in_client)):
            for y_idx in range(x_idx + 1, len(vars_in_client)):
                x, y = vars_in_client[x_idx], vars_in_client[y_idx]
                pair = compute_pair_cumulants(cdf, w, x, y)
                if pair:
                    client_cums.update(pair)
                    
        if client_cums:
            server_cumulants.append({"n_eff": n_eff, "cumulants": client_cums})
            
    total_neff = sum([c["n_eff"] for c in server_cumulants])
    agg_cums = {}
    for c_data in server_cumulants:
        wt = c_data["n_eff"] / total_neff
        for key, val in c_data["cumulants"].items():
            if key == "n_obs": continue
            agg_cums.setdefault(key, 0.0)
            agg_cums[key] += wt * val
            
    orders = []
    for sg in star_graphs:
        c, periph = sg["center"], sg["peripherals"]
        o = _recover(agg_cums, [c] + periph, ratio=0.05)
        if o: orders.append(o)
        
    G = _build_graph(orders)
    m = _metrics(G)
    
    print(f"  -> F1: {m['f1']:.3f}, SHD: {m['shd']}")
    results.append({"alpha": f"{int(alpha*11)}/11", "f1": m['f1'], "shd": m['shd']})

print("\n=== FINAL RESULTS FOR K=6 ===")
for r in results:
    print(f"Alpha = {r['alpha']}: F1 = {r['f1']:.3f}, SHD = {r['shd']}")

=== RUNNING ALL ALPHA DISTRIBUTIONS FOR K=6 ===

Running alpha = 0.18, Missingness: 0.1
  -> F1: 0.378, SHD: 23

Running alpha = 0.27, Missingness: 0.2
  -> F1: 0.370, SHD: 17

Running alpha = 0.36, Missingness: 0.3
  -> F1: 0.250, SHD: 18

Running alpha = 0.45, Missingness: 0.4
  -> F1: 0.182, SHD: 18

Running alpha = 0.55, Missingness: 0.5
  -> F1: 0.080, SHD: 23

=== FINAL RESULTS FOR K=6 ===
Alpha = 2/11: F1 = 0.378, SHD = 23
Alpha = 3/11: F1 = 0.370, SHD = 17
Alpha = 4/11: F1 = 0.250, SHD = 18
Alpha = 5/11: F1 = 0.182, SHD = 18
Alpha = 6/11: F1 = 0.080, SHD = 23
